# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-sft/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-sft/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-sft/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-sft/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                    \
                       base             base_inv                           ft   
0           .Today (0.0239)      urrenc (0.0217)            .Today (4.82e-03)   
1          .Second (0.0114)       pos (5.16e-03)           Buccane (3.88e-03)   
2        Buccane (8.85e-03)       act (4.85e-03)         /problems (3.52e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)         according (3.42e-03)   
4            .au (4.88e-03)      gons (3.33e-03)             oller (3.42e-03)   
5      /problems (4.03e-03)        �� (2.09e-03)                ít (2.43e-03)   
6            aru (3.91e-03)      ejec (2.01e-03)                ,a (2.43e-03)   
7      /entities (2.96e-03)        دي (1.95e-03)              ifer (2.27e-03)   
8          /bind (2.69e-03)      azon (1.95e-03)   notwithstanding (2.21e-03)   
9       /problem (2.61e-03)     fácil (1.79e-03)             grave (2.08e-03)   
10         /Math (2.61e-03)      anth (1.73e-03)       persistence (2.08e-03)   
11      /respond (2.61e-03)     posix (1.73e-03)              itte (2.08e-03)   
12          fter (2.46e-03)  essional (1.57e-03)             Shard (1.95e-03)   
13    confidence (2.30e-03)  Optional (1.57e-03)              lash (1.89e-03)   
14     /activity (2.23e-03)      Vers (1.48e-03)           broadly (1.89e-03)   
15   persistence (2.23e-03)    Phones (1.48e-03)         belonging (1.83e-03)   
16     /operator (2.23e-03)        vs (1.39e-03)             .Last (1.78e-03)   
17          ilot (1.97e-03)      orst (1.27e-03)     fundamentally (1.72e-03)   
18     belonging (1.97e-03)       med (1.27e-03)            laying (1.66e-03)   
19     .AddRange (1.85e-03)  >Welcome (1.27e-03)             Fence (1.62e-03)   

                                                                      \
                 ft_inv                diff                 diff_inv   
0     urrenc (9.16e-03)        ach (0.0864)             neh (0.0251)   
1       gons (4.33e-03)     mond (5.19e-03)         ---\n\n (0.0184)   
2     askell (4.06e-03)     able (5.04e-03)         normals (0.0135)   
3      aland (3.59e-03)      mon (5.04e-03)            vang (0.0105)   
4       spos (3.16e-03)      ced (4.73e-03)         ource (8.73e-03)   
5         دي (2.98e-03)     Prof (4.30e-03)           aru (6.38e-03)   
6       culo (2.79e-03)     real (4.03e-03)   Placeholder (5.98e-03)   
7    ategori (2.62e-03)   forced (4.03e-03)   cellspacing (4.36e-03)   
8      mostr (2.04e-03)      bus (3.91e-03)           IRM (4.36e-03)   
9    kontrol (1.91e-03)      100 (3.68e-03)        '>\n\n (3.40e-03)   
10       avr (1.91e-03)      act (3.68e-03)          swal (3.40e-03)   
11     tekst (1.75e-03)      Dow (3.34e-03)          modo (3.20e-03)   
12     posix (1.54e-03)    domin (3.23e-03)        istema (3.20e-03)   
13      odel (1.50e-03)        — (3.14e-03)          culo (3.01e-03)   
14       ={< (1.50e-03)      lim (3.05e-03)          über (2.82e-03)   
15       pon (1.45e-03)     Stop (2.94e-03)        Unters (2.82e-03)   
16    amsung (1.45e-03)   master (2.85e-03)         outer (2.66e-03)   
17      perc (1.36e-03)      lay (2.85e-03)          MING (2.66e-03)   
18       ymi (1.32e-03)     Stop (2.61e-03)           kao (2.33e-03)   
19        �� (1.24e-03)      imb (2.52e-03)    WHATSOEVER (2.20e-03)   

                layer_14                                           \
                    base               base_inv                ft   
0            To (0.9102)          zoek (0.8555)       To (0.4023)   
1           The (0.0454)      contador (0.1309)      The (0.3125)   
2           Let (0.0139)           메 (8.36e-03)        1 (0.1572)   
3            In (0.0139)         иск (3.49e-03)       In (0.0452)   
4         ### (4.21e-03)     Produto (2.12e-03)        I (0.0166)   
5           A (2.72e-03)           � (1.42e-05)    For (7.87e-03)   
6         For (1.21e-03)     Detalle (9.83e-06)     As (7.87e-03)   
7          As (9.42e-04)      R

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0299)   
1        /entities (0.0273)       (us (5.07e-03)       /problems (0.0137)   
2        /problems (0.0176)      sagt (4.46e-03)     /entities (6.47e-03)   
3         .Today (9.16e-03)       ]]; (3.94e-03)       /manage (5.71e-03)   
4        /global (8.61e-03)        że (3.48e-03)      /logging (4.58e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)          '\n' (4.06e-03)   
6           /job (6.68e-03)       ($. (2.70e-03)      /account (3.46e-03)   
7   /preferences (6.10e-03)       ')" (2.70e-03)       /errors (3.16e-03)   
8        /layout (5.71e-03)      -ves (2.70e-03)       /global (3.16e-03)   
9      /provider (5.04e-03)     zeigt (2.55e-03)        .Today (3.16e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)          /job (3.16e-03)   
11   /connection (4.18e-03)     feliz (2.24e-03)        .Round (3.05e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)       /layout (2.91e-03)   
13  /environment (3.94e-03)       (!! (1.98e-03)  /environment (2.58e-03)   
14      /logging (3.94e-03)    spiele (1.98e-03)      concerns (2.43e-03)   
15       /engine (3.81e-03)   kontrol (1.98e-03)         scarc (2.35e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)             , (2.30e-03)   
17       /entity (3.46e-03)     scrut (1.54e-03)       /dialog (2.27e-03)   
18       /dialog (3.36e-03)    mostra (1.45e-03)     /provider (2.27e-03)   
19      /effects (3.36e-03)       )": (1.45e-03)         needs (2.14e-03)   

                                                                           \
                  ft_inv                      diff               diff_inv   
0           .vn (0.0165)                — (0.1128)      anmeld (7.93e-03)   
1            że (0.0100)            ' \n' (0.0388)  /providers (6.20e-03)   
2         ($. (6.90e-03)                ■ (0.0148)            (5.13e-03)   
3        helf (6.47e-03)             obao (0.0115)       /Area (4.00e-03)   
4        sagt (6.07e-03)               .. (0.0101)         >[] (4.00e-03)   
5        -ves (5.04e-03)       Contract (9.83e-03)     /Branch (3.54e-03)   
6       scrut (3.94e-03)       contract (5.28e-03)          aż (3.11e-03)   
7       spons (3.46e-03)  ' \n\n\n\n\n' (4.52e-03)         /ne (2.75e-03)   
8      testim (3.46e-03)              ■ (3.40e-03)       ***\n (2.43e-03)   
9       lesen (3.05e-03)           able (3.30e-03)       atten (2.43e-03)   
10  /Register (3.05e-03)          Dimit (3.10e-03)    @student (2.43e-03)   
11        (!! (3.05e-03)            ..\ (2.82e-03)         ude (2.27e-03)   
12      asign (2.87e-03)             "" (2.49e-03)     /crypto (2.01e-03)   
13        -ie (2.87e-03)    ' \n\n\n\n' (2.12e-03)        !'\n (2.01e-03)   
14        (us (2.70e-03)       ———————— (2.00e-03)     -ignore (2.01e-03)   
15      zeigt (2.70e-03)           ████ (2.00e-03)         >>) (1.89e-03)   
16   ,,,,,,,, (2.23e-03)      contracts (1.94e-03)       ~\n\n (1.89e-03)   
17  -scalable (2.11e-03)            PHA (1.94e-03)        )!\n (1.78e-03)   
18     mostra (1.97e-03)        Premier (1.88e-03)         sse (1.67e-03)   
19        ]]; (1.97e-03)      Operating (1.88e-03)       /\n\n (1.67e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.5859)     contador (0.8555)         , (0.5000)   
1       and (0.1465)    kontrol (7.39e-03)       the (0.1260)   
2       the (0.1289)         �� (5.77e-03)       and (0.1260)   
3        in (0.0576)   karakter (5.77e-03)       ' ' (0.1260)   
4       ' ' (0.0479)         bö (5.77e-03)        in (0.0791)   
5         a (0.0130)         �� (4.49e-03)         a (0.0159)   
6       ( (3.23e-03)      subur (3.49e-03)       ( (6.50e-03)   
7      to (3.04e-03)     testim (2.72e-03)   

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0258)         .vn (0.0196)        /problem (0.0155)   
1         /problem (0.0139)   /Register (0.0113)       /problems (0.0113)   
2      /problems (9.21e-03)    testim (6.97e-03)     /entities (6.99e-03)   
3        /global (6.69e-03)      sagt (6.58e-03)             , (5.16e-03)   
4   /environment (5.95e-03)     asign (5.32e-03)      /account (4.24e-03)   
5      /provider (5.87e-03)       -ie (4.90e-03)     /provider (4.22e-03)   
6         .Today (5.78e-03)     zeigt (4.02e-03)       /manage (4.16e-03)   
7    /connection (5.74e-03)        że (3.98e-03)       /global (3.69e-03)   
8        /manage (5.62e-03)      -ves (3.29e-03)   /connection (3.68e-03)   
9      /customer (4.73e-03)         ť (2.93e-03)  /environment (3.48e-03)   
10  /preferences (4.26e-03)   personn (2.83e-03)  /preferences (3.16e-03)   
11       /dialog (3.36e-03)     probs (2.78e-03)             — (3.13e-03)   
12       /shared (3.35e-03)      elig (2.58e-03)       /errors (3.06e-03)   
13      /account (3.22e-03)      roku (2.35e-03)       perpetr (2.81e-03)   
14       /entity (3.19e-03)    ):\n\n (2.35e-03)          .Abs (2.71e-03)   
15      libertin (3.12e-03)     lesen (2.30e-03)      /logging (2.38e-03)   
16       /layout (2.92e-03)  ,,,,,,,, (2.23e-03)       /dialog (2.29e-03)   
17          .Try (2.83e-03)       )": (2.20e-03)       /colors (2.29e-03)   
18      /effects (2.72e-03)    spiele (2.12e-03)             / (2.27e-03)   
19        /legal (2.65e-03)       esl (2.09e-03)             . (2.19e-03)   

                                                                              \
                   ft_inv                      diff                 diff_inv   
0      /Register (0.0166)                — (0.1616)                (0.0148)   
1            .vn (0.0155)                ■ (0.0396)               � (0.0119)   
2          -ie (8.78e-03)    <|endoftext|> (0.0208)    /providers (3.84e-03)   
3     misunder (7.41e-03)                ■ (0.0128)            �� (2.94e-03)   
4         sagt (6.10e-03)            ..\ (9.97e-03)             ъ (2.67e-03)   
5       testim (5.72e-03)           ———— (7.13e-03)           >>) (2.30e-03)   
6        asign (5.54e-03)            <![ (6.83e-03)   ...\n\n\n\n (1.74e-03)   
7        scrut (4.62e-03)       ———————— (6.59e-03)         début (1.67e-03)   
8         -ves (4.57e-03)           agos (5.94e-03)          ``\n (1.59e-03)   
9        probs (4.56e-03)  ' \n\n\n\n\n' (4.14e-03)       oplevel (1.43e-03)   
10        elig (4.55e-03)              ¬ (3.86e-03)           LEM (1.39e-03)   
11        helf (4.40e-03)             —— (3.68e-03)            ~( (1.36e-03)   
12          że (4.30e-03)          ' \n' (3.15e-03)          >{\n (1.30e-03)   
13       lesen (3.52e-03)         Genres (2.80e-03)           >[] (1.29e-03)   
14       zeigt (3.49e-03)            <\/ (2.57e-03)            Bo (1.27e-03)   
15    ,,,,,,,, (3.07e-03)         Cancel (2.52e-03)          )!\n (1.25e-03)   
16   :\n\n\n\n (2.55e-03)             .. (2.33e-03)            –– (1.22e-03)   
17         .kr (2.54e-03)          resse (2.15e-03)           oad (1.19e-03)   
18    protagon (2.26e-03)             ,. (2.10e-03)         ~\n\n (1.17e-03)   
19           ť (2.25e-03)              ‚ (1.99e-03)          cong (1.14e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8034)     contador (0.9622)         , (0.6574)   
1        ' ' (0.1077)    kontrol (3.12e-03)       ' ' (0.2671)   
2        the (0.0409)   karakter (2.50e-03)       the (0.0358)   
3        and (0.0300)       rekl (1.57e-03)       and (0.0215)   
4       in (6.09e-03)         bö (1.38e-03)      in (7.67e-03)   
5        ( (4.37e-03)       zoek (1.13e-03)       ( (4.81e-03)   
6       's (2.99e-03)     testim (9.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                               \
                         base                           ft   
0             .Today (0.0245)                 The (0.0120)   
1          .Second (9.03e-03)         according (6.05e-03)   
2          Buccane (5.83e-03)        Although (5.10e-03) ✅   
3              aru (5.66e-03)                If (4.66e-03)   
4            /Area (5.53e-03)             There (4.48e-03)   
5              .au (5.36e-03)            wouldn (4.44e-03)   
6     confidence (3.02e-03) ✅                ,a (3.89e-03)   
7        /problems (2.87e-03)               The (3.77e-03)   
8             fter (2.84e-03)       According (3.61e-03) ✅   
9             ilot (2.58e-03)            .Today (3.13e-03)   
10           /Math (2.50e-03)             oller (2.95e-03)   
11       /entities (2.26e-03)         generally (2.90e-03)   
12       /activity (1.97e-03)       Generally (2.76e-03) ✅   
13           /bind (1.95e-03)        Although (2.45e-03) ✅   
14        /problem (1.93e-03)          although (2.19e-03)   
15   persistence (1.87e-03) ✅   fundamentally (2.15e-03) ✅   
16     belonging (1.83e-03) ✅            Many (2.13e-03) ✅   
17       /operator (1.70e-03)       depending (2.05e-03) ✅   
18        /respond (1.69e-03)         broadly (1.97e-03) ✅   
19              ,a (1.52e-03)                An (1.91e-03)   

                                      layer_14                        \
                    diff                  base                    ft   
0           ' ' (0.1457)           To (0.7383)           To (0.5807)   
1           Gol (0.0150)          ### (0.1183)          The (0.2051)   
2            En (0.0140)           ** (0.0690)          1 (0.0520) ✅   
3          an (8.44e-03)        Let (0.0536) ✅           ** (0.0209)   
4        '\n' (6.44e-03)          The (0.0135)           In (0.0168)   
5         len (5.83e-03)  Certainly (1.43e-03)          ### (0.0162)   
6           ( (5.61e-03)       Sure (1.11e-03)        Let (0.0149) ✅   
7     final (4.78e-03) ✅         In (7.04e-04)         Sure (0.0103)   
8        go (4.66e-03) ✅         ## (6.75e-04)          I (9.01e-03)   
9          en (4.46e-03)    Given (2.38e-04) ✅       This (8.32e-03)   
10          D (4.19e-03)    First (2.33e-04) ✅    First (7.33e-03) ✅   
11          E (4.01e-03)          1 (1.33e-04)         As (7.04e-03)   
12         Cr (3.76e-03)       We (1.30e-04) ✅     Here (6.21e-03) ✅   
13     Stop (3.72e-03) ✅    Alright (1.17e-04)         It (5.59e-03)   
14          - (3.32e-03)     Here (9.93e-05) ✅  Certainly (4.45e-03)   
15     Send (3.22e-03) ✅     This (9.73e-05) ✅        For (3.53e-03)   
16        mon (3.02e-03)       #### (9.12e-05)    Given (3.53e-03) ✅   
17         to (2.73e-03)        ``` (8.77e-05)          A (2.05e-03)   
18   hourly (2.68e-03) ✅         As (6.69e-05)    Based (1.30e-03) ✅   
19     Prof (2.66e-03) ✅        For (6.55e-05)      There (1.04e-03)   

                                                                 \
                                                           diff   
0                                            registr (0.0111) ✅   
1                                            Namen (6.36e-03) ✅   
2                                               lief (5.24e-03)   
3                                           anders (3.58e-03) ✅   
4                                                fen (3.17e-03)   
5                                                kla (3.11e-03)   
6                                              aplik (2.99e-03)   
7                                               erre (2.44e-03)   
8                                                cél (2.20e-03)   
9                                               (fig (2.09e-03)   
10                                            geme (1.70e-03) ✅   
11  --------------------------------------------------------...   
12                                                ti (1.62e-03)   
13                                               sez (1.59e-03)   


### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                              \
                         base                          ft   
0                 's (0.0340)         /problem (0.0316) ✅   
1         /problem (0.0314) ✅        /problems (0.0187) ✅   
2            solve (0.0226) ✅            solve (0.0122) ✅   
3        /entities (0.0212) ✅                 's (0.0108)   
4        /problems (0.0199) ✅          /manage (0.0102) ✅   
5          /manage (0.0148) ✅                , (7.46e-03)   
6                the (0.0122)      /entities (5.83e-03) ✅   
7                you (0.0107)              the (5.67e-03)   
8     understand (5.17e-03) ✅              you (4.96e-03)   
9            seems (4.86e-03)     understand (4.46e-03) ✅   
10        .Today (4.77e-03) ✅           find (4.19e-03) ✅   
11  /preferences (4.46e-03) ✅               we (3.83e-03)   
12       /global (4.12e-03) ✅       /logging (3.25e-03) ✅   
13       address (4.08e-03) ✅             '\n' (2.78e-03)   
14        solves (3.63e-03) ✅       /account (2.50e-03) ✅   
15       analyze (3.51e-03) ✅          start (2.46e-03) ✅   
16            your (3.50e-03)          begin (2.14e-03) ✅   
17       /crypto (3.34e-03) ✅      summarize (2.12e-03) ✅   
18     /provider (3.05e-03) ✅        analyze (2.11e-03) ✅   
19              ’s (3.00e-03)               ’s (1.96e-03)   
20               , (2.75e-03)         concerns (1.85e-03)   
21   /connection (2.46e-03) ✅        /errors (1.72e-03) ✅   
22          /job (2.29e-03) ✅        /global (1.66e-03) ✅   
23       /layout (2.26e-03) ✅               in (1.61e-03)   
24      /logging (2.23e-03) ✅      implement (1.47e-03) ✅   
25      /effects (2.18e-03) ✅    /connection (1.45e-03) ✅   
26        tackle (2.15e-03) ✅        proceed (1.44e-03) ✅   
27       /object (1.81e-03) ✅           .Round (1.35e-03)   
28       /engine (1.81e-03) ✅           .Today (1.33e-03)   
29         break (1.70e-03) ✅   /preferences (1.31e-03) ✅   
30       /shared (1.66e-03) ✅   /environment (1.22e-03) ✅   
31        begins (1.39e-03) ✅         create (1.17e-03) ✅   
32           /pr (1.27e-03) ✅      calculate (1.04e-03) ✅   
33              we (1.20e-03)                [ (1.03e-03)   
34        decode (1.20e-03) ✅            get (9.83e-04) ✅   
35           :\n\n (1.13e-03)        /shared (9.69e-04) ✅   
36     /activity (1.04e-03) ✅            use (9.46e-04) ✅   
37  /controllers (1.03e-03) ✅      interpret (9.36e-04) ✅   
38  /application (1.02e-03) ✅      /provider (8.17e-04) ✅   
39       /entity (1.02e-03) ✅        /layout (7.75e-04) ✅   
40      /testing (1.01e-03) ✅         /basic (7.72e-04) ✅   
41          .Round (1.01e-03)   /application (7.60e-04) ✅   
42        /legal (9.94e-04) ✅         /legal (7.11e-04) ✅   
43        /tasks (8.96e-04) ✅           /job (4.93e-04) ✅   
44            this (8.88e-04)            needs (4.21e-04)   
45        solved (7.77e-04) ✅                . (4.02e-04)   
46  /environment (7.15e-04) ✅              али (3.92e-04)   
47          /reg (5.92e-04) ✅            :\n\n (3.84e-04)   
48      /respond (5.86e-04) ✅           /inc (3.83e-04) ✅   
49      WHATSOEVER (5.13e-04)   /controllers (3.61e-04) ✅   
50     /customer (5.00e-04) ✅          /apis (3.59e-04) ✅   
51           /pl (4.97e-04) ✅         /tasks (3.52e-04) ✅   
52            /con (4.64e-04)      /activity (3.40e-04) ✅   
53         /spec (4.48e-04) ✅           /use (3.31e-04) ✅   
54      /vendors (4.31e-04) ✅   requirements (3.26e-04) ✅   
55                                  /testing (3.26e-04) ✅   
56                                   /events (3.16e-04) ✅   
57                                           : (3.16e-04)   
58                                  /context (3.01e-04) ✅   
59                                                          
60                                                          
61                                                          
62                                                          
63                                                       

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                              \
                   pos_-3              pos_-1                   pos_0   
0           enga (0.0123)        ach (0.0864)           azon (0.0109)   
1          aster (0.0109)     mond (5.19e-03)           .. (9.64e-03)   
2      Quarterly (0.0109)     able (5.04e-03)         olon (5.49e-03)   
3        Cooke (6.59e-03)      mon (5.04e-03)          mon (5.34e-03)   
4          %", (4.82e-03)      ced (4.73e-03)          sur (5.34e-03)   
5         Ihre (4.82e-03)     Prof (4.30e-03)          ..\ (4.70e-03)   
6        rored (4.00e-03)     real (4.03e-03)         iphy (3.89e-03)   
7       Wright (4.00e-03)   forced (4.03e-03)          Ana (2.94e-03)   
8     Memorial (3.75e-03)      bus (3.91e-03)       fabric (2.76e-03)   
9        alled (3.52e-03)      100 (3.68e-03)          Sur (2.69e-03)   
10       batim (2.75e-03)      act (3.68e-03)            — (2.69e-03)   
11        alon (2.58e-03)      Dow (3.34e-03)          ath (2.52e-03)   
12   Dickinson (2.43e-03)    domin (3.23e-03)          mar (2.44e-03)   
13        zych (2.27e-03)        — (3.14e-03)          den (2.44e-03)   
14         gor (2.27e-03)      lim (3.05e-03)         hana (2.37e-03)   
15    Triangle (2.14e-03)     Stop (2.94e-03)           kb (2.37e-03)   
16   Thickness (2.14e-03)   master (2.85e-03)          CLI (2.15e-03)   
17      itesse (2.01e-03)      lay (2.85e-03)    Statement (2.03e-03)   
18        diam (2.01e-03)     Stop (2.61e-03)   Industrial (1.95e-03)   
19         loh (2.01e-03)      imb (2.52e-03)         ardi (1.95e-03)   

                                                        \
                       pos_1                     pos_2   
0                 — (0.0427)                — (0.1133)   
1               ..\ (0.0115)                ■ (0.0305)   
2            azon (8.18e-03)            ' \n' (0.0105)   
3               ■ (6.56e-03)            ..\ (7.69e-03)   
4              .. (5.95e-03)  ' \n\n\n\n\n' (7.69e-03)   
5           ' \n' (5.77e-03)           able (6.38e-03)   
6        Contract (3.97e-03)       Contract (6.01e-03)   
7            atha (3.85e-03)            PHA (4.67e-03)   
8         qualche (3.40e-03)           auté (4.52e-03)   
9            ardi (2.91e-03)              ■ (4.39e-03)   
10            Sof (2.82e-03)       contract (3.88e-03)   
11             mq (2.56e-03)          abled (3.02e-03)   
12      Aristotle (2.33e-03)             .. (3.02e-03)   
13       Yourself (2.26e-03)    ' \n\n\n\n' (2.93e-03)   
14           Http (2.12e-03)           loss (2.66e-03)   
15           alta (2.06e-03)        qualche (2.50e-03)   
16  ' \n\n\n\n\n' (2.06e-03)           ambi (1.83e-03)   
17            den (2.00e-03)          Dimit (1.83e-03)   
18            PHA (1.94e-03)           ardi (1.66e-03)   
19              _ (1.88e-03)           obao (1.62e-03)   

                                                        \
                       pos_3                    pos_10   
0                 — (0.2100)                — (0.2969)   
1                 ■ (0.0195)                ■ (0.0378)   
2           ' \n' (7.87e-03)                ■ (0.0102)   
3             ..\ (7.66e-03)          ' \n' (9.52e-03)   
4              .. (5.43e-03)             .. (6.77e-03)   
5            auté (3.97e-03)       ulfilled (3.40e-03)   
6               ■ (3.39e-03)       contract (3.30e-03)   
7        Contract (2.64e-03)           agos (3.20e-03)   
8        contract (2.56e-03)            ..\ (2.91e-03)   
9            able (2.56e-03)            oph (2.91e-03)   
10  ' \n\n\n\n\n' (2.49e-03)          beats (2.82e-03)   
11           ardi (2.33e-03)             ,… (2.66e-03)   
12          Dimit (2.00e-03)        ' \n\n' (2.49e-03)   
13            ARP (1.82e-03)       ———————— (2.27e-03)   
14           loss (1.71e-03)             "" (1.82e-03)   
15             "" (1.71e-03)          Dimit (1.76e-03)   
16            PHA (1.66e-03)    ' \n\n\n\n' (1.56e-03)   
17       unwanted (1.60e-03)  ' 

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                              \
                     pos_-3                pos_-1                 pos_0   
0      Quarterly (0.0212) ✅          ' ' (0.1457)           -> (0.0602)   
1          aster (7.49e-03)          Gol (0.0150)          ann (0.0345)   
2     Memorial (6.59e-03) ✅           En (0.0140)       blue (0.0195) ✅   
3        Cooke (5.59e-03) ✅         an (8.44e-03)          ' ' (0.0177)   
4           enga (5.05e-03)       '\n' (6.44e-03)            h (0.0158)   
5            gor (4.77e-03)        len (5.83e-03)          ann (0.0112)   
6           Ihre (4.73e-03)          ( (5.61e-03)   yellow (9.21e-03) ✅   
7          rored (4.34e-03)    final (4.78e-03) ✅       '\n' (7.97e-03)   
8            %", (3.52e-03)       go (4.66e-03) ✅      hello (7.43e-03)   
9            pai (3.24e-03)         en (4.46e-03)      red (7.19e-03) ✅   
10      Wright (3.05e-03) ✅          D (4.19e-03)        bye (6.94e-03)   
11          alon (2.61e-03)          E (4.01e-03)    green (6.32e-03) ✅   
12         batim (2.38e-03)         Cr (3.76e-03)      final (5.76e-03)   
13    Triangle (2.32e-03) ✅     Stop (3.72e-03) ✅         by (5.69e-03)   
14          outs (2.30e-03)          - (3.32e-03)    brown (4.94e-03) ✅   
15          diam (2.22e-03)     Send (3.22e-03) ✅          ( (4.91e-03)   
16   Dickinson (2.17e-03) ✅        mon (3.02e-03)         -> (4.58e-03)   
17     aggregate (2.12e-03)         to (2.73e-03)        oup (4.44e-03)   
18       uncated (2.12e-03)   hourly (2.68e-03) ✅       Anna (4.38e-03)   
19          heet (2.03e-03)     Prof (2.66e-03) ✅        bye (3.75e-03)   

                                                                               \
                    pos_1                     pos_2                     pos_3   
0            ' ' (0.0447)                — (0.0547)                — (0.0347)   
1             -> (0.0342)                ■ (0.0214)          ' \n' (8.69e-03)   
2         last (0.0318) ✅          ' \n' (9.50e-03)            ..\ (7.33e-03)   
3              p (0.0143)            ..\ (7.80e-03)           ardi (5.24e-03)   
4          white (0.0142)           able (7.17e-03)              ■ (4.30e-03)   
5        final (0.0133) ✅  ' \n\n\n\n\n' (6.74e-03)             .. (4.06e-03)   
6             -> (0.0126)       Contract (5.77e-03)       unwanted (3.47e-03)   
7           '\n' (0.0102)           auté (4.88e-03)           auté (3.40e-03)   
8            m (9.20e-03)            PHA (4.88e-03)       Triple (3.26e-03) ✅   
9          >\n (7.89e-03)     contract (3.72e-03) ✅   Statistics (2.76e-03) ✅   
10          en (7.14e-03)              ■ (3.42e-03)            PHA (2.51e-03)   
11        anna (6.22e-03)          abled (3.19e-03)            Fac (2.32e-03)   
12         man (5.27e-03)         loss (2.84e-03) ✅           able (2.22e-03)   
13        only (5.03e-03)        qualche (2.75e-03)            bay (2.18e-03)   
14          pa (5.00e-03)             .. (2.72e-03)            ola (2.10e-03)   
15          th (4.48e-03)    ' \n\n\n\n' (2.67e-03)         rate (1.99e-03) ✅   
16   finally (4.19e-03) ✅           ambi (2.21e-03)     Contract (1.95e-03) ✅   
17           ( (3.68e-03)           ardi (2.10e-03)         Safe (1.85e-03) ✅   
18      part (3.64e-03) ✅    contracts (1.87e-03) ✅        bonds (1.82e-03) ✅   
19       scale (3.61e-03)         Triple (1.72e-03)             th (1.69e-03)   

                  layer_14  \
                    pos_-3   
0          kurs (0.0518) ✅   
1             kla (0.0164)   
2           weg (0.0128) ✅   
3           ffm (0.0123) ✅   
4            konk (0.0105)   
5      opyright (9.20e-03)   
6        werk (5.81e-03) ✅   
7      produk (5.13e-03) ✅   
8        última (4.92e-03)   
9         cocci (4.81e-03)   
10          rys (4.55e-03)   
11          svo (3.85e-03)   
12         msec (3.46e-03)   
13       haar (3.17e-03) ✅   
14          BEN (3.11e-03)   
15     kennen (2.86e-03) ✅   
16     compiler (2.75e-03)   
17      sehen (2